### Import Section

In [ ]:
import numpy as np 
import pandas as pd 

import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer

from sklearn.metrics import accuracy_score, recall_score, precision_score, classification_report

from xgboost import XGBClassifier
import joblib

In [ ]:

df = pd.read_csv(r'C:\Users\kulde\kuldeepcode\loan\loan_real_dataset.csv')

In [ ]:
df = df.drop(columns=['City', 'Job_Type', 'Education', 'Marital_Status'])

In [ ]:
# Feature Engineering
 
df["Risk_Level"] = df["Credit_Score"].apply(lambda x: "High" if x < 500 else "Low")


df["Age_Group"] = pd.cut(
    df["Age"],
    bins=[0, 18, 28, 50, 100],
    labels=['child',"Young", "Adult", "Old"]
)

df["Income_per_Loan"] = df["Salary"] / (df["Loan_Amount"] + 1)
df["Loan_to_Salary"] = df["Loan_Amount"] / (df["Salary"] + 1)
df["Total_Burden"] = df["Existing_Loans"] + df["EMI_Burden"]

In [ ]:
X = df.drop("Loan_Approved", axis=1)
y = df["Loan_Approved"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    stratify=y,
    random_state=42
)

In [ ]:
num_cols = X.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X.select_dtypes(include=['object', 'category', 'str']).columns

In [ ]:
processor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
])

In [ ]:
models = {
    "LogisticRegression": Pipeline([
        ("prep", processor),
        ("model", LogisticRegression(max_iter=300))
    ]),

    "RandomForest": Pipeline([
        ("prep", processor),
        ("model", RandomForestClassifier(
            n_estimators=300,
            random_state=42
        ))
    ]),

    "XGBoost": Pipeline([
        ("prep", processor),
        ("model", XGBClassifier(
            n_estimators=500,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42
        ))
    ])
}

In [ ]:
results = {}

for name, model in models.items():
    print(f"\n===== {name} =====")
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    results[name] = acc

    print("Accuracy:", acc)
    print(classification_report(y_test, y_pred))

In [ ]:
best_model_name = max(results, key=results.get)
print("\nBEST MODEL:", best_model_name)

In [ ]:
best_model = models[best_model_name]
print(best_model)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

xgb_base = XGBClassifier(
    random_state=42,
    eval_metric="logloss"
)

param_dist = {
    "n_estimators": [300, 500, 700],
    "max_depth": [3, 4, 5, 6, 8],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "subsample": [0.6, 0.7, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 1.0],
    "gamma": [0, 0.1, 0.2],
    "min_child_weight": [1, 3, 5]
}

search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_dist,
    n_iter=25,
    cv=3,
    scoring="accuracy",
    n_jobs=-1,
    verbose=2,
    random_state=42
)

In [ ]:
tuned_model = Pipeline([
    ("prep", processor),
    ("model", search)
])

In [ ]:
tuned_model.fit(X_train, y_train)

In [ ]:
y_pred = tuned_model.predict(X_test)

print("\n===== FINAL TUNED MODEL RESULTS =====")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [ ]:
print("\nBEST PARAMETERS:")
print(search.best_params_)

In [ ]:
cv_scores = cross_val_score(tuned_model, X, y, cv=5)

print("\nCV Scores:", cv_scores)
print("Mean CV:", cv_scores.mean())

In [ ]:
joblib.dump(tuned_model, "best_xgboost_loan_model.pkl")

In [ ]:
df.info()